# Phase 3: Demo Video (wicker table)

Renders the demo video from the Phase 2b model (`exports/phase2_scene1_mcmc_1M_30k.ply` on Drive). Nothing is trained here.

**Output** (in `results/phase3_demo/`):
| File | What |
|---|---|
| `01_input.mp4` | 4 s of the phone video, at 2× speed |
| `02_raw_vs_clean.mp4` | the model before and after `scripts/clean_splat.py`, side by side, same camera |
| `03_orbit.mp4` | a smooth orbit of the cleaned table, with full view-dependent colour (SH degree 3) |
| `demo.mp4` | the three clips joined: the demo video |
| `orbit.gif` | a small loop for the README |

The renders use the full model, so they look a little better than the web viewer, which keeps only the base colour.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Check the preview in section 4: if the table is badly framed, change `DIST` / `HEIGHT_DEG` and run from section 4 again.

⏱️ On a T4: install 2-5 min (the gsplat CUDA code loads from the Drive cache), cleanup ~1 min, rendering ~3-6 min.

## 1. GPU, Drive, settings

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, json, time, re, subprocess, shutil
SCENE   = 'scene1'
PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'
WORK    = '/content/work'
PLY     = f'{PROJECT}/exports/phase2_{SCENE}_mcmc_1M_30k.ply'     # written by notebook 02b, section 7
VIDEO   = f'{PROJECT}/videos/{SCENE}.mp4'
OUT     = f'{PROJECT}/results/phase3_demo'
for d in [WORK, OUT, f'{PROJECT}/cache']:
    os.makedirs(d, exist_ok=True)
assert os.path.exists(PLY), f'{PLY} not found: run notebook 02b (section 7 exports it)'

# video settings
W, H, FPS = 1280, 720, 30
VFOV_DEG   = 40          # vertical field of view
DIST       = 1.2         # camera distance from the table centre (scene units; the table is ~0.76 long)
HEIGHT_DEG = 28          # camera elevation above the floor
ORBIT_S    = 12          # orbit length in seconds (one full turn)
COMPARE_S  = 8           # raw-vs-clean clip length (a 120° arc)
BG = (13, 15, 18)        # background, same as the web viewer
METRICS = 'PSNR 32.15 dB · SSIM 0.965 · LPIPS 0.106 on held-out frames'

## 2. Install gsplat

The same two cells as notebooks 02 / 02b. The second one should print `✅ gsplat CUDA loaded from the Drive cache`.

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in torch.cuda.get_device_name(0):
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
# F1.5 (second fix): keep gsplat's compiled CUDA library on Drive and load it directly in later sessions
import torch, importlib.util
PREBUILT = '/content/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use /content/gsplat_prebuilt/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/content/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the Drive cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to Drive')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. Clean the model

Runs `scripts/clean_splat.py` (embedded below, same code as the repo): it levels the floor, crops a cylinder around the table, removes near-invisible, oversized and needle-shaped Gaussians, and splits the table from the scanned floor.

In [ ]:
%%writefile /content/clean_splat.py
"""Crop and clean a 3DGS .ply, and export a small .splat for the web viewer.

CPU only (numpy), so it runs on the laptop without touching the GPU.

What it does, in order:
1. Fits the floor plane (RANSAC on opaque Gaussians around the object) and builds a
   level frame: origin on the floor under the object, "up" pointing away from the floor.
2. Keeps Gaussians inside a cylinder around the object (radius, height above floor).
3. Drops near-invisible Gaussians (low opacity), huge ones, and needles
   (longest axis much longer than the second longest: the streaks seen in SuperSplat).
4. Writes the cleaned .ply (original coordinates, all SH kept, opens in SuperSplat)
   and a levelled, y-up .splat (32 bytes per Gaussian, base colour only) for the web viewer.

Example:
    python scripts/clean_splat.py "phase2_scene1_mcmc_1M_30k (1).ply" --out exports/table
"""
import argparse
import json
import os

import numpy as np

SH_C0 = 0.28209479177387814


# ---------- .ply I/O (binary little-endian, float properties, as gsplat writes them) ----------
def load_ply(path):
    with open(path, "rb") as f:
        header = b""
        while not header.endswith(b"end_header\n"):
            line = f.readline()
            if not line:
                raise ValueError("no end_header: not a .ply file")
            header += line
        text = header.decode("ascii")
        if "binary_little_endian" not in text:
            raise ValueError("only binary_little_endian .ply is supported")
        props = [l.split()[2] for l in text.splitlines() if l.startswith("property float")]
        n = int([l for l in text.splitlines() if l.startswith("element vertex")][0].split()[2])
        data = np.fromfile(f, dtype="<f4", count=n * len(props)).reshape(n, len(props))
    return props, data


def save_ply(path, props, data):
    header = "ply\nformat binary_little_endian 1.0\nelement vertex %d\n" % len(data)
    header += "".join(f"property float {p}\n" for p in props) + "end_header\n"
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        f.write(np.ascontiguousarray(data, dtype="<f4").tobytes())


# ---------- geometry helpers ----------
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def fit_plane_ransac(pts, iters=2000, thresh=0.01, seed=0):
    """Return (normal, d) with normal·p + d = 0, refined by least squares on the inliers."""
    rng = np.random.default_rng(seed)
    best, best_count = None, -1
    for _ in range(iters):
        p = pts[rng.choice(len(pts), 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0])
        if np.linalg.norm(n) < 1e-9:
            continue
        n /= np.linalg.norm(n)
        count = int((np.abs(pts @ n - n @ p[0]) < thresh).sum())
        if count > best_count:
            best, best_count = (n, -n @ p[0]), count
    n, d = best
    inl = pts[np.abs(pts @ n + d) < thresh]
    c = inl.mean(0)
    n = np.linalg.svd(inl - c)[2][-1]          # smallest singular vector = refined normal
    return n, -n @ c, len(inl)


def quat_from_matrix(R):
    """Rotation matrix -> quaternion (w, x, y, z)."""
    w = np.sqrt(max(0.0, 1 + R[0, 0] + R[1, 1] + R[2, 2])) / 2
    x = np.copysign(np.sqrt(max(0.0, 1 + R[0, 0] - R[1, 1] - R[2, 2])) / 2, R[2, 1] - R[1, 2])
    y = np.copysign(np.sqrt(max(0.0, 1 - R[0, 0] + R[1, 1] - R[2, 2])) / 2, R[0, 2] - R[2, 0])
    z = np.copysign(np.sqrt(max(0.0, 1 - R[0, 0] - R[1, 1] + R[2, 2])) / 2, R[1, 0] - R[0, 1])
    return np.array([w, x, y, z])


def quat_to_mats(q):
    """(N, 4) quaternions (w, x, y, z) -> (N, 3, 3) rotation matrices; column i is the axis of scale_i."""
    q = q / np.linalg.norm(q, axis=1, keepdims=True)
    w, x, y, z = q.T
    return np.stack([np.stack([1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)], 1),
                     np.stack([2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)], 1),
                     np.stack([2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)], 1)], 1)


def quat_mul(a, b):
    """Hamilton product a ⊗ b; a is (4,), b is (N, 4), both (w, x, y, z)."""
    w1, x1, y1, z1 = a
    w2, x2, y2, z2 = b.T
    return np.stack([w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
                     w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
                     w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
                     w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2], 1)


def level_frame(xyz, opacity, center_xy_radius, floor_rin, floor_rout):
    """Floor plane + object centre -> (R, origin): local = R @ (p - origin), local z = height above floor."""
    solid = opacity > 0.1
    # rough object centre: median of solid Gaussians, then the floor ring around it
    c0 = np.median(xyz[solid], 0)
    core = solid & (np.linalg.norm(xyz - c0, axis=1) < center_xy_radius)
    c0 = np.median(xyz[core], 0)
    # first guess of "up": the thinnest direction of the solid core (works for a floor-dominated scene)
    w, v = np.linalg.eigh(np.cov((xyz[core] - c0).T))
    up0 = v[:, 0]
    rel = xyz - c0
    radial = np.linalg.norm(rel - np.outer(rel @ up0, up0), axis=1)
    ring = solid & (radial > floor_rin) & (radial < floor_rout)
    # floor = the plane that most ring Gaussians lie on
    n, d, n_inl = fit_plane_ransac(xyz[ring])
    # point "up" from the floor towards the object
    if n @ c0 + d < 0:
        n, d = -n, -d
    origin = c0 - (n @ c0 + d) * n               # object centre dropped onto the floor
    ex = np.cross(n, [0.0, 0.0, 1.0] if abs(n[2]) < 0.9 else [1.0, 0.0, 0.0])
    ex /= np.linalg.norm(ex)
    ey = np.cross(n, ex)
    R = np.stack([ex, ey, n])                     # rows: local x, y, z(up)
    tilt = np.degrees(np.arccos(abs(n @ up0)))
    return R, origin, dict(floor_inliers=int(n_inl), floor_ring=int(ring.sum()),
                           tilt_vs_first_guess_deg=round(float(tilt), 2))


# ---------- export ----------
def write_splat(path, xyz, scales, rgb, alpha, quats):
    """antimatter15 .splat: per Gaussian 3f pos, 3f scale, 4u8 rgba, 4u8 rot (w,x,y,z)."""
    order = np.argsort(-(scales.prod(1) * alpha))   # big, opaque first (viewer convention)
    rec = np.zeros(len(xyz), dtype=[("p", "<f4", 3), ("s", "<f4", 3), ("c", "u1", 4), ("r", "u1", 4)])
    rec["p"] = xyz[order]
    rec["s"] = scales[order]
    rec["c"][:, :3] = np.clip(rgb[order] * 255, 0, 255).round()
    rec["c"][:, 3] = np.clip(alpha[order] * 255, 0, 255).round()
    q = quats[order] / np.linalg.norm(quats[order], axis=1, keepdims=True)
    rec["r"] = np.clip(q * 128 + 128, 0, 255).round()
    rec.tofile(path)


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("ply")
    ap.add_argument("--out", required=True, help="output path without extension (writes .ply, _object.splat, _floor.splat, .json)")
    ap.add_argument("--radius", type=float, default=0.75, help="keep within this distance of the object, along the floor")
    ap.add_argument("--max-height", type=float, default=0.75, help="keep up to this height above the floor")
    ap.add_argument("--below", type=float, default=0.03, help="keep this far below the floor plane (floor thickness)")
    ap.add_argument("--min-opacity", type=float, default=0.002,
                    help="MCMC builds thin structure and the floor from many faint Gaussians: keep this low")
    ap.add_argument("--max-scale", type=float, default=0.08, help="drop Gaussians whose longest axis is longer than this")
    ap.add_argument("--needle-ratio", type=float, default=8.0, help="longest / second-longest axis above this ...")
    ap.add_argument("--needle-min", type=float, default=0.04,
                    help="... and longest axis above this = needle (wicker strands are thin needles ~0.01-0.03: keep them)")
    ap.add_argument("--needle-above", type=float, default=0.03,
                    help="only remove needles higher than this above the floor (floor Gaussians are long and flat)")
    ap.add_argument("--floor-height", type=float, default=0.03,
                    help="flat, horizontal Gaussians below this height go to the separate _floor.splat")
    ap.add_argument("--object-ply", action="store_true", help="also write <out>_object.ply (object only, full SH)")
    ap.add_argument("--footprint-margin", type=float, default=0.03,
                    help="everything outside the object's footprint ellipse (+ this margin) goes to _floor.splat")
    ap.add_argument("--center-radius", type=float, default=0.3, help="radius used to find the object centre")
    ap.add_argument("--floor-ring", type=float, nargs=2, default=[0.45, 0.9], help="ring used to fit the floor")
    a = ap.parse_args()

    props, data = load_ply(a.ply)
    col = {p: i for i, p in enumerate(props)}
    xyz = data[:, [col["x"], col["y"], col["z"]]].astype(np.float64)
    opacity = sigmoid(data[:, col["opacity"]])
    scales = np.exp(data[:, [col["scale_0"], col["scale_1"], col["scale_2"]]])
    quats = data[:, [col["rot_0"], col["rot_1"], col["rot_2"], col["rot_3"]]].astype(np.float64)

    R, origin, frame_info = level_frame(xyz, opacity, a.center_radius, *a.floor_ring)
    local = (xyz - origin) @ R.T
    radial, height = np.hypot(local[:, 0], local[:, 1]), local[:, 2]

    s_sorted = np.sort(scales, 1)
    needle = ((s_sorted[:, 2] / np.maximum(s_sorted[:, 1], 1e-9) > a.needle_ratio) & (s_sorted[:, 2] > a.needle_min)
              & (height > a.needle_above))
    rules = {
        "outside cylinder": (radial > a.radius) | (height > a.max_height) | (height < -a.below),
        "low opacity": opacity < a.min_opacity,
        "too large": s_sorted[:, 2] > a.max_scale,
        "needle": needle,
    }
    keep = np.ones(len(data), bool)
    print(f"{len(data):,} Gaussians in {a.ply}")
    print(f"floor: {frame_info}")
    for name, m in rules.items():
        print(f"  {name:<17} {m.sum():>9,}  (removed by this rule after earlier ones: {(m & keep).sum():,})")
        keep &= ~m
    print(f"kept {keep.sum():,} ({keep.mean():.0%})")

    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    save_ply(a.out + ".ply", props, data[keep])

    # levelled, y-up copy for the web: local (x, y, up) -> three.js (x, up, -y)
    to_yup = np.array([[1, 0, 0], [0, 0, 1], [0, -1, 0]], float)
    M = to_yup @ R
    q_level = quat_from_matrix(M)
    rgb = np.clip(0.5 + SH_C0 * data[:, [col["f_dc_0"], col["f_dc_1"], col["f_dc_2"]]], 0, 1)
    pos_web = (xyz - origin) @ M.T
    quat_web = quat_mul(q_level, quats)

    # split off the scanned floor: flat Gaussians lying horizontally near the floor plane.
    # Object footprint: an ellipse fitted (on the floor plane) to the object's solid Gaussians above the floor.
    body = keep & (opacity > 0.1) & (height > a.floor_height + 0.05)
    fc = np.median(local[body][:, :2], 0)
    w, v = np.linalg.eigh(np.cov((local[body][:, :2] - fc).T))
    half = np.percentile(np.abs((local[body][:, :2] - fc) @ v), 99, axis=0)        # along v[:,0], v[:,1]
    uv = (local[:, :2] - fc) @ v
    inside = ((uv[:, 0] / (half[0] + a.footprint_margin)) ** 2 + (uv[:, 1] / (half[1] + a.footprint_margin)) ** 2) <= 1
    # Scanned floor = everything outside the footprint (floor and the wisps over it), plus the flat ground
    # layer under the object. Vertical leg Gaussians fail the "lying flat" test, so the table keeps its feet.
    mats = quat_to_mats(quats)
    thin_axis = mats[np.arange(len(data)), :, np.argmin(scales, 1)]          # direction of the smallest scale
    flat = (s_sorted[:, 0] < 0.3 * s_sorted[:, 1]) & (np.abs(thin_axis @ R[2]) > 0.8)
    floor = keep & (~inside | ((height < a.floor_height) & flat))
    obj = keep & ~floor
    write_splat(a.out + "_object.splat", pos_web[obj], scales[obj], rgb[obj], opacity[obj], quat_web[obj])
    write_splat(a.out + "_floor.splat", pos_web[floor], scales[floor], rgb[floor], opacity[floor], quat_web[floor])
    print(f"split: object {obj.sum():,} | scanned floor {floor.sum():,}")
    if a.object_ply:
        save_ply(a.out + "_object.ply", props, data[obj])          # full SH, original coordinates

    # for the viewer's plain floor: tile colour, and the footprint in web coordinates (x, z) = (local x, -local y)
    solid_floor = floor & (opacity > 0.05) & (height < a.floor_height)
    floor_rgb = np.median(rgb[solid_floor], 0) if solid_floor.sum() > 50 else np.array([0.8, 0.8, 0.8])
    flip = np.array([1.0, -1.0])
    footprint = dict(center=(fc * flip).round(4).tolist(), axis_long=(v[:, 1] * flip).round(4).tolist(),
                     half_long=round(float(half[1]), 4), half_short=round(float(half[0]), 4))
    low = obj & (opacity > 0.1)

    info = dict(source=os.path.basename(a.ply), total=int(len(data)), kept=int(keep.sum()),
                removed={k: int(v.sum()) for k, v in rules.items()}, params=vars(a), floor=frame_info,
                object_count=int(obj.sum()), floor_count=int(floor.sum()),
                floor_rgb=floor_rgb.round(3).tolist(), footprint=footprint,
                footprint_center_local=fc.round(4).tolist(),
                object_height=round(float(np.percentile(pos_web[low][:, 1], 99)), 3),
                origin=origin.round(5).tolist(), rotation_rows=R.round(6).tolist(),
                ply_mb=round(os.path.getsize(a.out + ".ply") / 1e6, 1),
                object_splat_mb=round(os.path.getsize(a.out + "_object.splat") / 1e6, 1),
                floor_splat_mb=round(os.path.getsize(a.out + "_floor.splat") / 1e6, 1))
    json.dump(info, open(a.out + ".json", "w"), indent=2)
    print(f"floor colour {info['floor_rgb']} | footprint {footprint}")
    print(f"wrote {a.out}.ply ({info['ply_mb']} MB), {a.out}_object.splat ({info['object_splat_mb']} MB), "
          f"{a.out}_floor.splat ({info['floor_splat_mb']} MB), {a.out}.json")


if __name__ == "__main__":
    main()

In [ ]:
CLEAN_OUT = f'{WORK}/table_clean'
r = subprocess.run([sys.executable, '/content/clean_splat.py', PLY, '--out', CLEAN_OUT, '--object-ply'],
                   capture_output=True, text=True)
print(r.stdout[-3000:], r.stderr[-3000:])
assert r.returncode == 0, 'clean_splat.py failed (see above)'
INFO = json.load(open(CLEAN_OUT + '.json'))
for f in [CLEAN_OUT + '.json', CLEAN_OUT + '_object.splat', CLEAN_OUT + '_floor.splat']:
    shutil.copy(f, OUT)      # the web viewer files, also kept on Drive
print('object', f"{INFO['object_count']:,}", '| floor', f"{INFO['floor_count']:,}")

## 4. Load the models and preview the camera

The camera orbits the table centre found by the cleanup script, with "up" = the floor normal, so the table stays level in the video.

In [ ]:
import numpy as np, torch, imageio.v2 as imageio
import matplotlib.pyplot as plt
from gsplat import rasterization
sys.path.insert(0, '/content')
import clean_splat as cs

def load_gaussians(path):
    props, d = cs.load_ply(path)
    col = {p: i for i, p in enumerate(props)}
    t = lambda names: torch.tensor(d[:, [col[n] for n in names]], dtype=torch.float32, device='cuda')
    n_rest = sum(p.startswith('f_rest_') for p in props) // 3
    sh0 = t(['f_dc_0', 'f_dc_1', 'f_dc_2'])[:, None, :]                                   # [N,1,3]
    shN = t([f'f_rest_{i}' for i in range(3 * n_rest)]).reshape(-1, 3, n_rest).transpose(1, 2)   # channel-major -> [N,K,3]
    return dict(means=t(['x', 'y', 'z']), quats=t(['rot_0', 'rot_1', 'rot_2', 'rot_3']),
                scales=torch.exp(t(['scale_0', 'scale_1', 'scale_2'])),
                opacities=torch.sigmoid(t(['opacity'])[:, 0]),
                colors=torch.cat([sh0, shN], 1), sh_degree=int(round((n_rest + 1) ** 0.5)) - 1)

RAW   = load_gaussians(PLY)
CLEAN = load_gaussians(CLEAN_OUT + '_object.ply')
print(f"raw {len(RAW['means']):,} | clean table {len(CLEAN['means']):,} | SH degree {CLEAN['sh_degree']}")

R_lvl  = np.array(INFO['rotation_rows'])                 # rows: floor x, floor y, up (world coordinates)
ORIGIN = np.array(INFO['origin'])
UP     = R_lvl[2]
fc     = INFO['footprint_center_local']
TARGET = ORIGIN + R_lvl.T @ np.array([fc[0], fc[1], 0.2])   # look at the table centre, 0.2 above the floor

def camera(angle_deg, dist=None, height_deg=None):
    # world-to-camera matrix (OpenCV: x right, y down, z forward) for a point on the orbit
    dist = DIST if dist is None else dist
    el, az = np.radians(HEIGHT_DEG if height_deg is None else height_deg), np.radians(angle_deg)
    pos = TARGET + dist * (np.cos(el) * (np.cos(az) * R_lvl[0] + np.sin(az) * R_lvl[1]) + np.sin(el) * UP)
    z = TARGET - pos; z /= np.linalg.norm(z)
    y = -(UP - (UP @ z) * z); y /= np.linalg.norm(y)
    x = np.cross(y, z)
    Rw = np.stack([x, y, z])
    V = np.eye(4); V[:3, :3] = Rw; V[:3, 3] = -Rw @ pos
    return torch.tensor(V, dtype=torch.float32, device='cuda')

BG_T = torch.tensor(BG, dtype=torch.float32, device='cuda') / 255

@torch.no_grad()
def render(g, viewmat, w=W, h=H):
    f = 0.5 * h / np.tan(np.radians(VFOV_DEG) / 2)
    K = torch.tensor([[f, 0, w / 2], [0, f, h / 2], [0, 0, 1]], dtype=torch.float32, device='cuda')
    img, _, _ = rasterization(g['means'], g['quats'], g['scales'], g['opacities'], g['colors'],
                              viewmat[None], K[None], w, h, sh_degree=g['sh_degree'], backgrounds=BG_T[None])
    return (img[0].clamp(0, 1).cpu().numpy() * 255).astype(np.uint8)

fig, ax = plt.subplots(1, 3, figsize=(18, 4))
for a, ang in zip(ax, [0, 120, 240]):
    a.imshow(render(CLEAN, camera(ang))); a.set_title(f'clean, {ang}°'); a.axis('off')
plt.show()

## 5. Render the clips

Every clip is 1280×720, 30 fps, H.264, with a caption, so they can be joined without re-encoding.

In [ ]:
from PIL import Image, ImageDraw, ImageFont
def font(size):
    for p in ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', '/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf']:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()
F_BIG, F_SMALL = font(30), font(20)

def caption(frame, title, sub=None, corner_labels=None):
    im = Image.fromarray(frame); d = ImageDraw.Draw(im, 'RGBA')
    d.rectangle([0, H - (92 if sub else 62), W, H], fill=(0, 0, 0, 150))
    d.text((28, H - (82 if sub else 50)), title, font=F_BIG, fill=(240, 240, 240))
    if sub:
        d.text((28, H - 40), sub, font=F_SMALL, fill=(190, 190, 190))
    for x, text in (corner_labels or []):
        d.text((x, 18), text, font=F_BIG, fill=(240, 240, 240), stroke_width=3, stroke_fill=(0, 0, 0))
    return np.asarray(im)

def write_clip(path, frames):
    # pipe raw RGB frames into ffmpeg: H.264, yuv420p (plays everywhere)
    p = subprocess.Popen(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'rawvideo', '-pix_fmt', 'rgb24', '-s', f'{W}x{H}',
                          '-r', str(FPS), '-i', '-', '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-crf', '18',
                          '-preset', 'medium', path], stdin=subprocess.PIPE)
    n = 0
    for fr in frames:
        p.stdin.write(np.ascontiguousarray(fr).tobytes()); n += 1
    p.stdin.close(); assert p.wait() == 0, f'ffmpeg failed on {path}'
    print(f'{os.path.basename(path)}: {n} frames, {n / FPS:.1f} s, {os.path.getsize(path) / 1e6:.1f} MB')

ease = lambda t: 0.5 - 0.5 * np.cos(np.pi * t)       # slow start and end

# 01: the input video, 4 s at 2x speed
tmp = f'{WORK}/input_frames'; shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
subprocess.run(['ffmpeg', '-loglevel', 'error', '-ss', '2', '-i', VIDEO, '-t', '8', '-vf',
                f'setpts=0.5*PTS,fps={FPS},scale={W}:{H}:force_original_aspect_ratio=decrease,'
                f'pad={W}:{H}:(ow-iw)/2:(oh-ih)/2:color=0x0d0f12', f'{tmp}/%04d.png'], check=True)
write_clip(f'{OUT}/01_input.mp4', (caption(imageio.imread(f), 'Input: ~1 min phone video', 'shown at 2x speed')
                                   for f in sorted(glob.glob(f'{tmp}/*.png'))))

# 02: raw vs clean, same camera, side by side
t0 = time.time(); n = COMPARE_S * FPS
def compare_frames():
    for i in range(n):
        v = camera(-30 + 120 * ease(i / (n - 1)), dist=DIST * 1.35)
        both = np.concatenate([render(RAW, v, W // 2, H), render(CLEAN, v, W // 2, H)], 1)
        yield caption(both, 'Cleanup: crop, drop faint / huge / needle Gaussians',
                      f"scripts/clean_splat.py: {INFO['total']:,} → {INFO['object_count']:,} Gaussians (table)",
                      [(28, 'Raw'), (W // 2 + 28, 'Cleaned')])
write_clip(f'{OUT}/02_raw_vs_clean.mp4', compare_frames())
print(f'  rendered in {time.time() - t0:.0f} s')

# 03: orbit of the cleaned table, full SH
t0 = time.time(); n = ORBIT_S * FPS
orbit = []
def orbit_frames():
    for i in range(n):
        fr = render(CLEAN, camera(360 * i / n, height_deg=HEIGHT_DEG + 6 * np.sin(2 * np.pi * i / n)))
        orbit.append(fr)
        yield caption(fr, '3D Gaussian Splatting twin, rendered in real time', METRICS)
write_clip(f'{OUT}/03_orbit.mp4', orbit_frames())
print(f'  rendered in {time.time() - t0:.0f} s')

## 6. Join the clips, make the README GIF

In [ ]:
with open(f'{WORK}/concat.txt', 'w') as f:
    for c in ['01_input.mp4', '02_raw_vs_clean.mp4', '03_orbit.mp4']:
        f.write(f"file '{OUT}/{c}'\n")
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'concat', '-safe', '0', '-i', f'{WORK}/concat.txt',
                '-c', 'copy', f'{OUT}/demo.mp4'], check=True)
print(f"demo.mp4: {os.path.getsize(f'{OUT}/demo.mp4') / 1e6:.1f} MB")

# GIF: the orbit without captions, 480 px wide, 15 fps, 2-pass palette for clean colours
gif_src = f'{WORK}/orbit_plain.mp4'
write_clip(gif_src, orbit)
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', gif_src, '-vf',
                'fps=15,scale=480:-1:flags=lanczos,split[a][b];[a]palettegen=max_colors=128[p];[b][p]paletteuse=dither=bayer:bayer_scale=4',
                f'{OUT}/orbit.gif'], check=True)
print(f"orbit.gif: {os.path.getsize(f'{OUT}/orbit.gif') / 1e6:.1f} MB")
imageio.imwrite(f'{OUT}/hero.jpg', orbit[len(orbit) // 8], quality=92)

from IPython.display import Video, display
display(Video(f'{OUT}/demo.mp4', embed=True, width=800))

## 7. Download

In [ ]:
DOWNLOAD = True      # demo.mp4, orbit.gif, hero.jpg and the two web viewer files
if DOWNLOAD:
    from google.colab import files
    for f in ['demo.mp4', 'orbit.gif', 'hero.jpg']:
        files.download(f'{OUT}/{f}')

## ✅ Phase 3 checklist
- [ ] Section 4 preview: table centred and level
- [ ] `demo.mp4` plays; the raw-vs-clean clip shows the streaks removed
- [ ] `orbit.gif` under ~10 MB (GitHub shows it in the README)
- [ ] Put `orbit.gif` and `hero.jpg` in `docs/images/` and `demo.mp4` on YouTube / LinkedIn

**Paste back into the chat:** the section 3 output and a screenshot of the section 4 preview, or tell me what looks off.